# Q3 - Dropout

In [ ]:
import os, sys
if os.path.exists('/kaggle/working/ML_Assignment'):
    os.system('cd /kaggle/working/ML_Assignment && git pull')
else:
    os.system('git clone https://github.com/AvtandilSh1/ML_Assignment.git /kaggle/working/ML_Assignment')
sys.path.insert(0, '/kaggle/working/ML_Assignment')
os.chdir('/kaggle/working/ML_Assignment')

In [ ]:
import urllib.request, tarfile, os
datasets_dir = '/kaggle/working/ML_Assignment/cs231n/datasets'
os.makedirs(datasets_dir, exist_ok=True)
cifar_dir = os.path.join(datasets_dir, 'cifar-10-batches-py')
if not os.path.exists(cifar_dir):
    tar_path = os.path.join(datasets_dir, 'cifar.tar.gz')
    urllib.request.urlretrieve('http://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz', tar_path)
    with tarfile.open(tar_path, 'r:gz') as t:
        t.extractall(datasets_dir)
    os.remove(tar_path)

In [ ]:
import numpy as np
from cs231n.layers import *
from cs231n.classifiers.fc_net import *
from cs231n.data_utils import get_CIFAR10_data
from cs231n.gradient_check import eval_numerical_gradient, eval_numerical_gradient_array
from cs231n.solver import Solver

def rel_error(x, y):
    return np.max(np.abs(x - y) / (np.maximum(1e-8, np.abs(x) + np.abs(y))))

## Dropout Forward - შემოწმება

In [ ]:
np.random.seed(231)
x = np.random.randn(500, 500) + 10

for p in [0.25, 0.4, 0.7]:
    out, _ = dropout_forward(x, {'mode': 'train', 'p': p})
    out_test, _ = dropout_forward(x, {'mode': 'test', 'p': p})
    print(f'p={p} | train mean: {out.mean():.4f} | test mean: {out_test.mean():.4f} | zeros: {(out==0).mean():.4f}')

## Dropout Backward - Gradient Check

In [ ]:
np.random.seed(231)
x = np.random.randn(10, 10) + 10
dout = np.random.randn(*x.shape)
dropout_param = {'mode': 'train', 'p': 0.2, 'seed': 123}
out, cache = dropout_forward(x, dropout_param)
dx = dropout_backward(dout, cache)
dx_num = eval_numerical_gradient_array(lambda xx: dropout_forward(xx, dropout_param)[0], x, dout)
print('dx relative error (expect <1e-10):', rel_error(dx, dx_num))

## FullyConnectedNet + Dropout - Gradient Check

In [ ]:
np.random.seed(231)
N, D, H1, H2, C = 2, 15, 20, 30, 10
X = np.random.randn(N, D)
y = np.random.randint(C, size=(N,))

for p in [1, 0.75, 0.5]:
    print(f'dropout_keep_ratio={p}')
    model = FullyConnectedNet([H1, H2], input_dim=D, num_classes=C,
                              weight_scale=5e-2, dtype=np.float64,
                              dropout_keep_ratio=p, seed=123)
    loss, grads = model.loss(X, y)
    print(f'  loss: {loss:.4f}')
    for name in sorted(grads):
        f = lambda _: model.loss(X, y)[0]
        grad_num = eval_numerical_gradient(f, model.params[name], verbose=False, h=1e-5)
        print(f'  {name}: {rel_error(grad_num, grads[name]):.2e}')

## Dropout vs No Dropout - სწავლება

In [ ]:
np.random.seed(231)
data = get_CIFAR10_data()
small_data = {
    'X_train': data['X_train'][:500],
    'y_train': data['y_train'][:500],
    'X_val':   data['X_val'],
    'y_val':   data['y_val'],
}

solvers = {}
for p in [1, 0.25]:
    model = FullyConnectedNet([500], dropout_keep_ratio=p)
    solver = Solver(model, small_data, num_epochs=25, batch_size=100,
                    update_rule='adam', optim_config={'learning_rate': 5e-4}, verbose=False)
    solver.train()
    solvers[p] = solver
    print(f'dropout_keep={p} | train_acc={max(solver.train_acc_history):.3f} | val_acc={max(solver.val_acc_history):.3f}')

## MLflow - DagsHub-ზე Logging

In [ ]:
!pip install -q mlflow
import mlflow, os, logging, contextlib, io
logging.getLogger('mlflow').setLevel(logging.ERROR)
os.environ['MLFLOW_TRACKING_USERNAME'] = 'ashos22'
os.environ['MLFLOW_TRACKING_PASSWORD'] = '78b25562699413e85d83414b742443e9df7c5cb5'

with contextlib.redirect_stdout(io.StringIO()):
    mlflow.set_tracking_uri('https://dagshub.com/ashos22/ML_Assignment.mlflow')
    mlflow.set_experiment('Q3-Dropout')
    for p, solver in solvers.items():
        with mlflow.start_run(run_name=f'dropout_keep_{p}'):
            mlflow.log_param('dropout_keep_ratio', p)
            mlflow.log_param('hidden_dims', '[500]')
            mlflow.log_param('num_epochs', 25)
            mlflow.log_param('n_train', 500)
            mlflow.log_metrics({
                'best_val_acc':   float(max(solver.val_acc_history)),
                'best_train_acc': float(max(solver.train_acc_history)),
            })

print('Logged to DagsHub.')